In [1]:
import cv2
from pathlib import Path
import pandas as pd
import numpy as np
from skimage.draw import polygon
import matplotlib.pyplot as plt
import sys
from scipy import stats as scipy_stats

In [2]:
# now for ejection fraction
# path_ef_ens = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/ensemble/results_ensemble_ef_adults.csv'
# path_ef_phiseg = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/phiseg/results_phiseg_ef_adults.csv'
# path_ef_vids = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/vids/results_vids_ef_adults_3.csv'

path_ef_ens = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/ens_comb/results_ensemble_ef_adults_norm.csv'
path_ef_phiseg = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/phiseg_comb/results_phiseg_ef_norm.csv'
path_ef_vids = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/vids_comb/results_vids_ef_norm.csv'

# combined augmentation
# path_ef_ens = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/ens_comb/results_ensemble_ef_adults_aug.csv'
# path_ef_phiseg = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/phiseg_comb/results_phiseg_ef_augm.csv'
# path_ef_vids = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/vids_comb/results_vids_ef_augm.csv'

df_ef_ens = pd.read_csv(path_ef_ens)
df_ef_phiseg = pd.read_csv(path_ef_phiseg)
df_ef_vids = pd.read_csv(path_ef_vids)

In [3]:
# compute uncertainty-relared measures for EF


bins = [-np.inf, 0, 2, 5, 12, 18]
labels = ['infant', 'toddler', 'preschooler', 'school age', 'teenager']

# bins = [-np.inf, 12, 18]
# labels = ['children', 'teenager']

df_ef_ens['age_group'] = pd.cut(df_ef_ens['age'], bins=bins, labels=labels, right=True)

# Absolute error
df_ef_ens['abs_error'] = np.abs(df_ef_ens['mean_ef_pred'] - df_ef_ens['gt_ef'])

# Uncertainty interval: [mean_ef_pred - std_ef_pred, mean_ef_pred + std_ef_pred]
# Coverage: does gt_ef fall inside this interval?
df_ef_ens['covered'] = (
    (df_ef_ens['gt_ef'] >= df_ef_ens['mean_ef_pred'] - df_ef_ens['std_ef_pred']) &
    (df_ef_ens['gt_ef'] <= df_ef_ens['mean_ef_pred'] + df_ef_ens['std_ef_pred'])
)

# Interval size = 2 * std_ef_pred
df_ef_ens['interval_size'] = 2 * df_ef_ens['std_ef_pred']


df_ef_phiseg['age_group'] = pd.cut(df_ef_phiseg['age'], bins=bins, labels=labels, right=True)

# Absolute error
df_ef_phiseg['abs_error'] = np.abs(df_ef_phiseg['mean_ef_pred'] - df_ef_phiseg['gt_ef'])

# Uncertainty interval: [mean_ef_pred - std_ef_pred, mean_ef_pred + std_ef_pred]
# Coverage: does gt_ef fall inside this interval?
df_ef_phiseg['covered'] = (
    (df_ef_phiseg['gt_ef'] >= df_ef_phiseg['mean_ef_pred'] - df_ef_phiseg['std_ef_pred']) &
    (df_ef_phiseg['gt_ef'] <= df_ef_phiseg['mean_ef_pred'] + df_ef_phiseg['std_ef_pred'])
)

# Interval size = 2 * std_ef_pred
df_ef_phiseg['interval_size'] = 2 * df_ef_phiseg['std_ef_pred']


df_ef_vids['age_group'] = pd.cut(df_ef_vids['age'], bins=bins, labels=labels, right=True)

# Absolute error
df_ef_vids['abs_error'] = np.abs(df_ef_vids['mean_ef_pred'] - df_ef_vids['gt_ef'])

# Uncertainty interval: [mean_ef_pred - std_ef_pred, mean_ef_pred + std_ef_pred]
# Coverage: does gt_ef fall inside this interval?
df_ef_vids['covered'] = (
    (df_ef_vids['gt_ef'] >= df_ef_vids['mean_ef_pred'] - df_ef_vids['std_ef_pred']) &
    (df_ef_vids['gt_ef'] <= df_ef_vids['mean_ef_pred'] + df_ef_vids['std_ef_pred'])
)

# Interval size = 2 * std_ef_pred
df_ef_vids['interval_size'] = 2 * df_ef_vids['std_ef_pred']

In [4]:
# df_ef_vids.head()

In [5]:
def compute_uncertainty_stats(group):
    result = {}

    # Coverage
    result['coverage'] = group['covered'].mean()

    # Interval size stats
    result['mean_interval_size'] = group['interval_size'].mean()
    result['std_interval_size'] = group['interval_size'].std()

    # Correlation between interval size and absolute error
    result['pearson_r'] = group['interval_size'].corr(group['abs_error'])
    # if len(group) > 2:
    #     spearman_r, spearman_p = scipy_stats.spearmanr(group['interval_size'], group['abs_error'])
    #     result['spearman_r'] = spearman_r
    #     result['spearman_p'] = spearman_p
    # else:
    #     result['spearman_r'] = np.nan
    #     result['spearman_p'] = np.nan

    # result['n_samples'] = len(group)
    return pd.Series(result)

In [6]:
stats_ensemble = df_ef_ens.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
stats_phiseg = df_ef_phiseg.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
stats_vids = df_ef_vids.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
print(stats_ensemble.round(3))
print(stats_phiseg.round(3))
print(stats_vids.round(3))

             coverage  mean_interval_size  std_interval_size  pearson_r
age_group                                                              
infant          0.083              25.872             44.271      0.989
toddler         0.345               7.912              3.547      0.779
preschooler     0.333               7.765              3.563      0.205
school age      0.190               7.449              7.453      0.345
teenager        0.252               8.094              3.801      0.683
             coverage  mean_interval_size  std_interval_size  pearson_r
age_group                                                              
infant          0.250              12.564             14.622      0.982
toddler         0.276               7.008              2.772      0.581
preschooler     0.286               6.553              2.149      0.157
school age      0.220               5.989              2.351      0.447
teenager        0.229               7.379              3.194    

/tmp/ipykernel_147696/45506707.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats_ensemble = df_ef_ens.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
/tmp/ipykernel_147696/45506707.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats_phiseg = df_ef_phiseg.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
/tmp/ipykernel_147696/45506707.py:3: DeprecationW